# RNA-FM

In [ ]:
import os
import pandas as pd
import time
import subprocess
from pathlib import Path

In [ ]:
# Clone RNA-FM if needed
rnafm_dir = Path.cwd().parent / "tools" / "RNA-FM"
if not rnafm_dir.exists():
    print("Cloning RNA-FM source code...")
    os.makedirs(str(rnafm_dir.parent), exist_ok=True)
    os.chdir(str(rnafm_dir.parent))
    !git clone https://github.com/ml4bio/RNA-FM.git
    os.chdir(str(Path.cwd()))
    print("RNA-FM cloned successfully")
else:
    print(f"RNA-FM already exists at {rnafm_dir}")

In [ ]:
method_name = "RNA-FM"
base = Path.cwd()
rnafm_dir = base.parent / "tools" / "RNA-FM"
rnafm_env = rnafm_dir / "rnafm-env"
redevelop_dir = rnafm_dir / "redevelop"

print(f"Base directory: {base}")
print(f"RNA-FM directory: {rnafm_dir}")
print(f"Redevelop directory: {redevelop_dir}")
print(f"RNA-FM environment: {rnafm_env}")

In [ ]:
# Create RNA-FM conda environment if needed
if not rnafm_env.exists():
    print("Creating RNA-FM conda environment...")
    os.chdir(str(rnafm_dir))
    !conda env create -f environment.yml --prefix {rnafm_env} -y
    os.chdir(str(base))
    print("RNA-FM environment created successfully")
else:
    print(f"RNA-FM environment already exists")

In [ ]:
# Install huggingface_hub
%pip install -q huggingface_hub
%pip install -q pyyaml

In [ ]:
# Download RNA-FM model weights from HuggingFace
from huggingface_hub import snapshot_download

model_dir = redevelop_dir / "pretrained" / "Models"

if not model_dir.exists():
    print("Downloading RNA-FM model weights from HuggingFace...")
    os.makedirs(str(model_dir), exist_ok=True)
    
    local_dir = snapshot_download(
        repo_id="cuhkaih/rnafm",
        repo_type="model",
        local_dir=str(model_dir),
        local_dir_use_symlinks=False
    )
    
    print(f"Model weights downloaded successfully to: {local_dir}")
else:
    print(f"Model weights already exist at: {model_dir}")

In [ ]:
def read_virus_fasta(path: str):
    lines = [ln.strip() for ln in open(path, 'r').read().splitlines() if ln.strip() != '']
    records = []
    for i in range(0, len(lines), 3):
        header, seq, struct = lines[i], lines[i+1], lines[i+2]
        name = header[1:].strip()
        records.append((name, seq.strip(), struct.strip()))
    df = pd.DataFrame(records, columns=['name','sequence','structure']).set_index('name')
    return df

viruses = read_virus_fasta('../data/viruses.fasta')

selected_virus_keys = None

if selected_virus_keys is None:
    virus_ids = list(viruses.index)
else:
    tmp = []
    for k in selected_virus_keys:
        if isinstance(k, int):
            tmp.append(viruses.index[k])
        else:
            tmp.append(str(k))
    virus_ids = tmp

In [ ]:
def run_rnafm_prediction(input_fasta, output_dir="rnafm_temp_results"):
    """Run RNA-FM prediction on a fasta file"""
    os.makedirs(output_dir, exist_ok=True)
    
    original_dir = os.getcwd()
    os.chdir(redevelop_dir)
    
    cmd = [
        "conda", "run", "-p", str(rnafm_env), "python", "launch/predict.py",
        "--config=pretrained/ss_prediction.yml",
        f"--data_path={original_dir}/{input_fasta}",
        f"--save_dir={original_dir}/{output_dir}",
        "--save_frequency", "1",
        "--gpu_id", "1"
    ]
    result = subprocess.run(cmd, capture_output=True, text=True)
    os.chdir(original_dir)
    
    return output_dir

In [ ]:
import shutil

out_fasta_name = method_name
output_dir_base = base.parent / "prediction"
os.makedirs(output_dir_base, exist_ok=True)
output_fasta = output_dir_base / (out_fasta_name + ".fasta")

if output_fasta.exists():
    os.remove(output_fasta)

print(f"{' ':3}\t{'virus':<20}\t{'len':<5}\t{'time'}")

for i, vid in enumerate(virus_ids):
    start_time = time.time()
    seq = viruses.loc[vid]['sequence']
    print(f"{i+1:3d}/{len(virus_ids)}\t{vid:<20}\t{len(seq):<5}\t", end='', flush=True)

    input_fasta = f"rnafm_input_{i}.fasta"
    with open(input_fasta, "w") as ofile:
        ofile.write(f">{vid}\n{seq}\n")

    temp_output_dir = run_rnafm_prediction(input_fasta, f"rnafm_temp_{i}")
    elapsed_time = time.time() - start_time

    ct_files = list(Path(temp_output_dir).rglob("*.ct"))
    
    if not ct_files:
        print(f"{elapsed_time: .1f} s [fail]")
        os.remove(input_fasta)
        shutil.rmtree(temp_output_dir, ignore_errors=True)
        continue
    
    ct_file = ct_files[0]
    dot_file = f"tmp_{vid}.dot"
    
    convert_cmd = f"python {base.parent}/methods/ct2dot.py {ct_file} {dot_file} -f full -q"
    convert_result = subprocess.run(convert_cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
    
    if convert_result.returncode != 0:
        print(f"{elapsed_time: .1f} s [fail]")
        os.remove(input_fasta)
        shutil.rmtree(temp_output_dir, ignore_errors=True)
        if os.path.exists(dot_file):
            os.remove(dot_file)
        continue
    
    structure = None
    if os.path.exists(dot_file):
        with open(dot_file) as f:
            lines = f.readlines()
            if len(lines) >= 3:
                structure = lines[2].strip()
    
    if structure is None:
        print(f"{elapsed_time: .1f} s [fail]")
        os.remove(input_fasta)
        shutil.rmtree(temp_output_dir, ignore_errors=True)
        if os.path.exists(dot_file):
            os.remove(dot_file)
        continue
    
    try:
        with open(ct_file, 'r') as f:
            lines = f.readlines()
        predicted_seq = ''.join([parts[1] for parts in [line.strip().split() for line in lines[1:] if line.strip()] if len(parts) >= 2])
        
        with open(output_fasta, "a") as out_f:
            out_f.write(f">{vid}\n")
            out_f.write(f"{predicted_seq}\n")
            out_f.write(f"{structure}\n")

        print(f"{elapsed_time: .1f} s")

        os.remove(input_fasta)
        os.remove(dot_file)
        shutil.rmtree(temp_output_dir, ignore_errors=True)
    except Exception as e:
        print(f"{elapsed_time: .1f} s [fail]")
        os.remove(input_fasta)
        shutil.rmtree(temp_output_dir, ignore_errors=True)
        if os.path.exists(dot_file):
            os.remove(dot_file)